<a href="https://colab.research.google.com/github/evanskips2727-del/CodeAlpha_WebScraping./blob/main/Web_scraping_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import requests

headers = {"User-Agent": "Mozilla/5.0 (educational project; CodeAlpha internship)"}

sites = {
    "Books to Scrape": "https://books.toscrape.com/",
    "WebScraper test e-commerce": "https://webscraper.io/test-sites/e-commerce/static/computers/laptops",
    "Quotes to Scrape": "https://quotes.toscrape.com/",
    "The Star Kenya": "https://www.the-star.co.ke/",
    "Citizen Digital": "https://www.citizen.digital/",
    "Kenyans.co.ke": "https://www.kenyans.co.ke/",
    "Standard Media": "https://www.standardmedia.co.ke/",
    "BrighterMonday": "https://www.brightermonday.co.ke/jobs",
    "PigiaMe": "https://www.pigiame.co.ke/",
}

for name, url in sites.items():
    try:
        r = requests.get(url, headers=headers, timeout=15)
        blocked = "Just a moment" in r.text or r.status_code in (403, 429)
        print(f"{name:28} {r.status_code}  {'BLOCKED' if blocked else 'OK'}  ({len(r.text)} chars)")
    except Exception as e:
        print(f"{name:28} ERROR {type(e).__name__}")

Books to Scrape              200  OK  (51294 chars)
WebScraper test e-commerce   200  OK  (37834 chars)
Quotes to Scrape             200  OK  (11021 chars)
The Star Kenya               200  OK  (362930 chars)
Citizen Digital              200  OK  (531435 chars)
Kenyans.co.ke                200  OK  (88985 chars)
Standard Media               200  OK  (497995 chars)
BrighterMonday               200  OK  (702511 chars)
PigiaMe                      403  BLOCKED  (133587 chars)


In [ ]:

import requests

headers = {"User-Agent": "Mozilla/5.0 (educational project; CodeAlpha internship)"}

sites = {
    "Kilimall Kenya": "https://www.kilimall.co.ke/",
    "Jiji Kenya": "https://jiji.co.ke/",
    "Carrefour Kenya": "https://www.carrefour.ke/",
    "Naivas": "https://naivas.online/",
    "Quickmart": "https://quickmart.co.ke/",
    "Masoko": "https://www.masoko.com/",
    "Phone Place Kenya": "https://www.phoneplacekenya.com/",
    "Gadgets Kenya": "https://gadgetskenya.com/",
}

for name, url in sites.items():
    try:
        r = requests.get(url, headers=headers, timeout=15)
        blocked = "Just a moment" in r.text or r.status_code in (403, 429)
        shopify = "cdn.shopify.com" in r.text
        woo = "woocommerce" in r.text.lower()
        print(f"{name:20} {r.status_code}  {'BLOCKED' if blocked else 'OK':8} "
              f"shopify={shopify} woo={woo} ({len(r.text)} chars)")
    except Exception as e:
        print(f"{name:20} ERROR {type(e).__name__}")

Kilimall Kenya       200  OK       shopify=False woo=False (170708 chars)
Jiji Kenya           403  BLOCKED  shopify=False woo=False (5465 chars)
Carrefour Kenya      403  BLOCKED  shopify=False woo=False (370 chars)
Naivas               403  BLOCKED  shopify=False woo=False (4835 chars)
Quickmart            200  OK       shopify=False woo=False (89995 chars)
Masoko               200  OK       shopify=False woo=False (301539 chars)
Phone Place Kenya    403  BLOCKED  shopify=False woo=False (5520 chars)
Gadgets Kenya        200  OK       shopify=False woo=False (26115 chars)


In [ ]:

import requests, re, json
from bs4 import BeautifulSoup

headers = {"User-Agent": "Mozilla/5.0 (educational project; CodeAlpha internship)"}

sites = {
    "Kilimall": "https://www.kilimall.co.ke",
    "Quickmart": "https://quickmart.co.ke",
    "Masoko": "https://www.masoko.com",
    "Gadgets Kenya": "https://gadgetskenya.com",
}

for name, base in sites.items():
    print("=" * 65)
    print(name)

    # 1. robots.txt
    try:
        rb = requests.get(base + "/robots.txt", headers=headers, timeout=15)
        print("robots.txt:", rb.status_code)
        print(rb.text[:600])
    except Exception as e:
        print("robots.txt error:", type(e).__name__)

    # 2. homepage signals
    try:
        r = requests.get(base, headers=headers, timeout=15)
        html = r.text
        soup = BeautifulSoup(html, "html.parser")
        print("\nHomepage:", r.status_code, f"({len(html)} chars)")
        print("JSON-LD blocks      :", len(soup.find_all("script", type="application/ld+json")))
        print("__NUXT__ / __NEXT__ :", "__NUXT__" in html, "/", "__NEXT_DATA__" in html)
        print("Price mentions      :", len(re.findall(r"(?:KES|Ksh|KSh)\.?\s?[\d,]+", html, flags=re.I)))
        print("Product-like links  :", len([a for a in soup.find_all("a", href=True)
              if any(k in a["href"] for k in ["/product", "/item", "/p/", "/shop"])]))
        print("Shopify / Woo       :", "cdn.shopify.com" in html, "/", "woocommerce" in html.lower())
    except Exception as e:
        print("Homepage error:", type(e).__name__)

    # 3. structured product feeds
    for path in ["/products.json", "/wp-json/wc/store/products?per_page=2"]:
        try:
            p = requests.get(base + path, headers=headers, timeout=15)
            ok = p.status_code == 200 and p.text.strip().startswith(("{", "["))
            print(f"Feed {path:42} {p.status_code} {'JSON OK' if ok else 'no'}")
        except Exception as e:
            print(f"Feed {path:42} error {type(e).__name__}")

Kilimall
robots.txt: 200
# Allow all user agents (crawlers) to access all pages of the site
User-agent: *

# Block access to specific directories
Disallow: /protocol/**.*
Disallow: /api/*
Disallow: /login?*
Disallow: /user-center*
Disallow: /shopping-cart
Disallow: /reviews-publish*
Disallow: /orders*
Disallow: /checkout
Disallow: /voucher
Disallow: /coins
Disallow: /address*
Disallow: /setting*
Disallow: /item*
Disallow: /deepLink*
Disallow: /skuPrice*
Disallow: /product*
Disallow: /new/goods*
Disallow: /404
Disallow: /newUserUrl*
Disallow: /player-video*

# Specify the location of the sitemap
Sitemap: https://www.ki

Homepage: 200 (170465 chars)
JSON-LD blocks      : 1
__NUXT__ / __NEXT__ : True / False
Price mentions      : 12
Product-like links  : 0
Shopify / Woo       : False / False
Feed /products.json                             200 no
Feed /wp-json/wc/store/products?per_page=2      200 no
Quickmart
robots.txt: 404
<!DOCTYPE html>
<html lang="en" prefix="og: http://ogp.me/ns#"
 

In [ ]:

import requests, json
from bs4 import BeautifulSoup

headers = {"User-Agent": "Mozilla/5.0 (educational project; CodeAlpha internship)"}

# --- Masoko: look inside __NEXT_DATA__ ---
r = requests.get("https://www.masoko.com", headers=headers, timeout=15)
soup = BeautifulSoup(r.text, "html.parser")
tag = soup.find("script", id="__NEXT_DATA__")
data = json.loads(tag.string)

print("Top keys:", list(data.keys()))
print("pageProps keys:", list(data.get("props", {}).get("pageProps", {}).keys()))

def walk(o, path="root"):
    if isinstance(o, dict):
        if any("price" in k.lower() for k in o):
            print("\nFirst price-like record at:", path)
            print({k: o[k] for k in list(o)[:10]})
            return True
        for k, v in o.items():
            if walk(v, f"{path}.{k}"):
                return True
    elif isinstance(o, list):
        for i, v in enumerate(o[:3]):
            if walk(v, f"{path}[{i}]"):
                return True
    return False

if not walk(data):
    print("\nNo price-like fields found in the homepage JSON")

# --- Links from both sites ---
for name, base in [("Masoko", "https://www.masoko.com"), ("Gadgets Kenya", "https://gadgetskenya.com")]:
    rr = requests.get(base, headers=headers, timeout=15)
    s = BeautifulSoup(rr.text, "html.parser")
    links = sorted({a["href"] for a in s.find_all("a", href=True) if a["href"].count("/") >= 1})
    print("\n" + "=" * 50)
    print(name, "-", len(links), "unique links")
    for l in links[:25]:
        print(l)

Top keys: ['props', 'page', 'query', 'buildId', 'isFallback', 'isExperimentalCompile', 'dynamicIds', 'gsp', 'scriptLoader']
pageProps keys: ['data', 'flags']

First price-like record at: root.props.pageProps.data.newWidgets.sections[2].data.product_list_section.products[0]
{'id': 83590, 'sku': 'DEVPlug879-burgundy', 'name': "iPhone 18 Pro Max 5G: 6.9'' - 12GB RAM - 256GB ROM - Burgundy", 'typeId': 'simple', 'price': '283794', 'thumbnail': 'https://assets.os.masoko.com/media/catalog/product/i/p/iphone_18_pro_max_burgundy_2.jpg', 'urlPath': 'iphone-18-pro-max-5g-6-9-12gb-ram-256gb-rom-burgundy', 'specialPrice': None, 'deviceInsuranceMode': 'opt_out', 'deviceInsurancePercent': '6'}

Masoko - 41 unique links
/
/apple-devices
/categories
/computing/routers
/customer-service?firstLoad=true
/fall-in-love-with-this-month-s-latest-tech
/for-all-the-love-buds-out-there
/for-the-love-of-the-game
/gaming-items
/hidden-gems
/laptops
/laptops/hp
/phones-accessories/accessories
/phones-accessories/ac

In [ ]:

import requests, json
from bs4 import BeautifulSoup

headers = {"User-Agent": "Mozilla/5.0 (educational project; CodeAlpha internship)"}
BASE = "https://www.masoko.com"

def get_next_data(path):
    r = requests.get(BASE + path, headers=headers, timeout=15)
    print(path, "->", r.status_code)
    soup = BeautifulSoup(r.text, "html.parser")
    tag = soup.find("script", id="__NEXT_DATA__")
    return json.loads(tag.string) if tag else None

def find_product_lists(o, path="root", found=None):
    if found is None:
        found = []
    if isinstance(o, dict):
        for k, v in o.items():
            find_product_lists(v, f"{path}.{k}", found)
    elif isinstance(o, list):
        if o and isinstance(o[0], dict) and "price" in o[0] and "name" in o[0]:
            found.append((path, o))
        else:
            for i, v in enumerate(o):
                find_product_lists(v, f"{path}[{i}]", found)
    return found

# 1. Category page
data = get_next_data("/phones-accessories/mobile-phones")
lists = find_product_lists(data)
for path, items in lists:
    print("\nProduct list at:", path, "| items:", len(items))
    print("Fields:", list(items[0].keys()))

# 2. Pagination hints
def find_keys(o, hints=("total", "page", "limit", "size", "count", "last"), path="root"):
    if isinstance(o, dict):
        for k, v in o.items():
            if any(h in k.lower() for h in hints) and not isinstance(v, (dict, list)):
                print(f"{path}.{k} = {v}")
            find_keys(v, hints, f"{path}.{k}")
    elif isinstance(o, list):
        for i, v in enumerate(o[:2]):
            find_keys(v, hints, f"{path}[{i}]")

print("\nPagination-like keys:")
find_keys(data.get("props", {}).get("pageProps", {}))
print("Query:", data.get("query"))

# 3. Does ?page=2 return different products?
d2 = get_next_data("/phones-accessories/mobile-phones?page=2")
l2 = find_product_lists(d2)
if lists and l2:
    ids1 = [p["id"] for p in lists[0][1]]
    ids2 = [p["id"] for p in l2[0][1]]
    print("\nPage1 first ids:", ids1[:5])
    print("Page2 first ids:", ids2[:5])
    print("Pagination works:", ids1[:5] != ids2[:5])

# 4. One product page: look for rating / brand / reviews
first = lists[0][1][0]
pd = get_next_data("/" + first["urlPath"])
print("\nProduct page pageProps keys:", list(pd["props"]["pageProps"].keys()))
txt = json.dumps(pd).lower()
for kw in ["rating", "review", "brand", "stock", "description", "category"]:
    print(f"{kw:12} mentioned:", kw in txt)

/phones-accessories/mobile-phones -> 200

Product list at: root.props.pageProps.category.categoryPage.items | items: 20
Fields: ['id', 'sku', 'name', 'typeId', 'price', 'thumbnail', 'urlPath', 'maxPrice', 'specialPrice', 'vendorName', 'rating', 'likes', 'deviceFinancing', 'deviceInsuranceMode', 'deviceInsurancePercent', 'stock', 'configurableOptions', 'campaign', '__typename']

Pagination-like keys:
root.category.categoryPage.pagination.currentPage = 1
root.category.categoryPage.pagination.totalCount = 299
root.category.categoryPage.pagination.pageSize = 20
root.category.categoryPage.pagination.nextPage = 2
root.category.categoryPage.cmsPage = None
Query: {'category_path': ['phones-accessories', 'mobile-phones']}
/phones-accessories/mobile-phones?page=2 -> 200

Page1 first ids: [79877, 80418, 80511, 80512, 80744]
Page2 first ids: [81892, 81893, 81894, 81896, 81898]
Pagination works: True
/tecno-camon-20-pro-5g-6-67-8gb-ram-256gb-rom-b -> 404

Product page pageProps keys: []
rating     

In [ ]:

import requests, json, time, random
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime

headers = {"User-Agent": "Mozilla/5.0 (educational project; CodeAlpha internship)"}
BASE = "https://www.masoko.com"

CATEGORIES = [
    "phones-accessories/mobile-phones",
    "phones-accessories/accessories",
    "laptops",
    "apple-devices",
    "gaming-items",
    "computing/routers",
]

def fetch_category_page(path, page):
    url = f"{BASE}/{path}" + (f"?page={page}" if page > 1 else "")
    for attempt in range(3):
        try:
            r = requests.get(url, headers=headers, timeout=20)
            if r.status_code == 200:
                soup = BeautifulSoup(r.text, "html.parser")
                tag = soup.find("script", id="__NEXT_DATA__")
                if tag:
                    cp = json.loads(tag.string)["props"]["pageProps"]["category"]["categoryPage"]
                    return cp
            else:
                print(f"  {url} -> {r.status_code} (attempt {attempt+1})")
        except Exception as e:
            print(f"  error {type(e).__name__} (attempt {attempt+1})")
        time.sleep(3)
    return None

all_rows = []
for cat in CATEGORIES:
    print(f"\nScraping: {cat}")
    page = 1
    while page:
        cp = fetch_category_page(cat, page)
        if not cp or not cp.get("items"):
            print(f"  page {page}: no data, stopping this category")
            break
        for item in cp["items"]:
            item["category"] = cat
            item["page"] = page
            item["scraped_at"] = datetime.now().isoformat(timespec="seconds")
            all_rows.append(item)
        pag = cp.get("pagination", {})
        print(f"  page {page}/{-(-pag.get('totalCount', 0)//pag.get('pageSize', 20))} "
              f"| total so far: {len(all_rows)}")
        page = pag.get("nextPage")
        time.sleep(random.uniform(1.5, 2.5))

df = pd.json_normalize(all_rows)
before = len(df)
df = df.drop_duplicates(subset="id", keep="first")
print(f"\nRaw rows: {before} | after dedupe by id: {len(df)}")

# Lists/dicts -> JSON strings so the CSV is clean
for col in df.columns:
    if df[col].apply(lambda x: isinstance(x, (list, dict))).any():
        df[col] = df[col].apply(lambda x: json.dumps(x) if isinstance(x, (list, dict)) else x)

df.to_csv("masoko_products_raw.csv", index=False)
print("Saved masoko_products_raw.csv")
print(df.shape)
print(df["category"].value_counts())


Scraping: phones-accessories/mobile-phones
  page 1/15 | total so far: 20
  page 2/15 | total so far: 40
  page 3/15 | total so far: 60
  page 4/15 | total so far: 80
  page 5/15 | total so far: 100
  page 6/15 | total so far: 120
  page 7/15 | total so far: 140
  page 8/15 | total so far: 160
  page 9/15 | total so far: 180
  page 10/15 | total so far: 200
  page 11/15 | total so far: 220
  page 12/15 | total so far: 240
  error ReadTimeout (attempt 1)
  page 13/15 | total so far: 260
  error ReadTimeout (attempt 1)
  page 14/15 | total so far: 280
  page 15/15 | total so far: 299

Scraping: phones-accessories/accessories
  page 1/37 | total so far: 319
  page 2/37 | total so far: 339
  page 3/37 | total so far: 359
  page 4/37 | total so far: 379
  page 5/37 | total so far: 399
  page 6/37 | total so far: 419
  page 7/37 | total so far: 439
  page 8/37 | total so far: 459
  page 9/37 | total so far: 479
  page 10/37 | total so far: 499
  page 11/37 | total so far: 519
  page 12/37 |

In [ ]:

print(df.dtypes)
print(df.head(3).T)
print(df.isna().sum().sort_values(ascending=False).head(15))

id                         int64
sku                       object
name                      object
typeId                    object
price                     object
thumbnail                 object
urlPath                   object
maxPrice                  object
specialPrice              object
vendorName                object
rating                    object
likes                     object
deviceFinancing           object
deviceInsuranceMode       object
deviceInsurancePercent    object
configurableOptions       object
campaign                  object
__typename                object
category                  object
page                       int64
scraped_at                object
stock.isInStock             bool
stock.qty                  int64
stock.__typename          object
dtype: object
                                                                        0  \
id                                                                  79877   
sku                                     

In [ ]:

from google.colab import drive
drive.mount("/content/drive")
import shutil, os
os.makedirs("/content/drive/MyDrive/CodeAlpha", exist_ok=True)
shutil.copy("masoko_products_raw.csv", "/content/drive/MyDrive/CodeAlpha/")

Mounted at /content/drive


'/content/drive/MyDrive/CodeAlpha/masoko_products_raw.csv'

In [ ]:

import pandas as pd, numpy as np, re, os, shutil

path = "masoko_products_raw.csv"
if not os.path.exists(path):
    path = "/content/drive/MyDrive/CodeAlpha/masoko_products_raw.csv"
df = pd.read_csv(path)
print("Loaded:", df.shape)

# 1. Drop columns confirmed empty/constant
empty_cols = ["maxPrice", "likes", "vendorName", "rating", "deviceFinancing",
              "deviceInsurancePercent", "configurableOptions", "__typename", "stock.__typename"]
df = df.drop(columns=empty_cols, errors="ignore")
df = df.rename(columns={"stock.isInStock": "in_stock", "stock.qty": "stock_qty"})

# 2. Numeric prices + discount features
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df["specialPrice"] = pd.to_numeric(df["specialPrice"], errors="coerce")
df["price_anomaly"] = (df["price"] <= 0) | (df["specialPrice"] >= df["price"])
df["has_discount"] = df["specialPrice"].notna() & (df["specialPrice"] < df["price"])
df["final_price"] = np.where(df["has_discount"], df["specialPrice"], df["price"])
df["discount_kes"] = np.where(df["has_discount"], df["price"] - df["specialPrice"], 0)
df["discount_pct"] = (df["discount_kes"] / df["price"] * 100).round(2)

# 3. Brand from product name
APPLE = {"iphone", "ipad", "macbook", "airpods", "apple", "imac", "ipod"}
def get_brand(name):
    first = re.split(r"[\s:\-]+", str(name).strip())[0].lower()
    return "Apple" if first in APPLE else first.title()
df["brand"] = df["name"].apply(get_brand)

# 4. Specs from product name
def first_num(pattern, s):
    m = re.search(pattern, str(s), flags=re.I)
    return float(m.group(1)) if m else np.nan

def storage(s):
    m = re.search(r"(\d+)\s*(GB|TB)\s*(?:ROM|SSD|HDD|STORAGE)", str(s), flags=re.I)
    if not m:
        return np.nan
    v = float(m.group(1))
    return v * 1024 if m.group(2).upper() == "TB" else v

df["ram_gb"] = df["name"].apply(lambda s: first_num(r"(\d+)\s*GB\s*RAM", s))
df["storage_gb"] = df["name"].apply(storage)
df["screen_in"] = df["name"].apply(lambda s: first_num(r"(\d+(?:\.\d+)?)\s*(?:''|\"|”|inch)", s))

# 5. Misc features
df["category_group"] = df["category"].str.split("/").str[-1]
df["has_campaign"] = df["campaign"].fillna("[]").astype(str).ne("[]")

# 6. Quality report
print("\nShape:", df.shape)
print("\nMissing values:\n", df.isna().sum()[lambda s: s > 0])
print("\nFinal price (KES):\n", df["final_price"].describe().round(0))
print(f"\nDiscounted: {df['has_discount'].sum()} ({df['has_discount'].mean():.1%})")
print("Discount % summary:\n", df.loc[df["has_discount"], "discount_pct"].describe().round(2))
print("\nTop brands:\n", df["brand"].value_counts().head(12))
print("\nSpec coverage (non-null):")
print(df[["ram_gb", "storage_gb", "screen_in"]].notna().sum())
print("\nPrice anomalies:", df["price_anomaly"].sum())

# 7. Save locally and to Drive
df.to_csv("masoko_products_clean.csv", index=False)
os.makedirs("/content/drive/MyDrive/CodeAlpha", exist_ok=True)
shutil.copy("masoko_products_clean.csv", "/content/drive/MyDrive/CodeAlpha/")
print("\nSaved masoko_products_clean.csv")

Loaded: (1293, 24)

Shape: (1293, 26)

Missing values:
 thumbnail                 1
specialPrice            974
deviceInsuranceMode    1292
ram_gb                  878
storage_gb              899
screen_in               857
dtype: int64

Final price (KES):
 count      1293.0
mean      65904.0
std       93534.0
min          94.0
25%        5200.0
50%       17600.0
75%       89995.0
max      646758.0
Name: final_price, dtype: float64

Discounted: 319 (24.7%)
Discount % summary:
 count    319.00
mean      15.79
std       15.37
min        0.10
25%        5.41
50%       10.22
75%       18.23
max       72.42
Name: discount_pct, dtype: float64

Top brands:
 brand
Apple       291
Jbl         112
Samsung      67
Hama         55
Havit        51
Xiaomi       45
Logitech     41
Sony         40
Canyon       40
Oraimo       36
Nothing      36
Marshall     30
Name: count, dtype: int64

Spec coverage (non-null):
ram_gb        415
storage_gb    394
screen_in     436
dtype: int64

Price anomalies: 0

Sa

In [ ]:

import pandas as pd, numpy as np, os, shutil

df = pd.read_csv("masoko_products_clean.csv")

# 1. Brand formatting
special = {"Jbl": "JBL", "Hp": "HP", "Lg": "LG", "Jvc": "JVC", "Tcl": "TCL",
           "Msi": "MSI", "Zte": "ZTE", "Oppo": "OPPO", "Oneplus": "OnePlus"}
def fix_brand(b):
    b = str(b)
    if b in special:
        return special[b]
    return b.upper() if len(b) <= 3 else b
df["brand"] = df["brand"].apply(fix_brand)

# Drop constant column
df = df.drop(columns=["deviceInsuranceMode"], errors="ignore")

short = lambda s: s.astype(str).str[:55]

print("=== 10 cheapest items ===")
c = df.nsmallest(10, "final_price")[["name", "final_price"]].copy()
c["name"] = short(c["name"]); print(c.to_string(index=False))

print("\n=== 5 biggest discounts ===")
d = df.nlargest(5, "discount_pct")[["name", "price", "specialPrice", "discount_pct"]].copy()
d["name"] = short(d["name"]); print(d.to_string(index=False))

print("\n=== Spec coverage by category (share non-null) ===")
print(df.groupby("category_group")[["ram_gb", "storage_gb", "screen_in"]]
        .apply(lambda g: g.notna().mean().round(2)))

print("\n=== Brands ===")
vc = df["brand"].value_counts()
print("Unique brands:", len(vc))
print(vc.tail(25).to_string())

print("\n=== Stock ===")
print("In stock share:", df["in_stock"].mean().round(3))
print(df["stock_qty"].describe().round(1))

print("\n=== Duplicate names:", df.duplicated("name").sum())

df.to_csv("masoko_products_clean.csv", index=False)
shutil.copy("masoko_products_clean.csv", "/content/drive/MyDrive/CodeAlpha/")
print("\nSaved.")

=== 10 cheapest items ===
                                        name  final_price
              Vision Plus Micro Usb Cable 1m         94.0
Vision Plus Stereo Earphones (Extra Comfort)        127.0
      Oraimo OCD-M53 2A Data cable IM Black         130.0
      Oraimo Data Cable OCD-114M 2A 1M Black        135.0
                 Vision Plus Type C Cable 1M        138.0
      Oraimo Data Cable OCD-114C 2A 1M Black        180.0
Tecno Omni Fast High Speed Data Cable TCDM13        200.0
             Oraimo OCD-C53 USB cable black         205.0
      Oraimo Data Cable OCD-114L 2A 1M Black        205.0
            Vision Plus Type C to C Cable 1M        226.0

=== 5 biggest discounts ===
                                              name  price  specialPrice  discount_pct
                     Vision Plus Cool Pods - Black 2295.0         633.0         72.42
                     Vision Plus Cool Pods - White 2295.0         633.0         72.42
      Vision Plus Stereo Earphones (Extra Comfort

In [ ]:

import pandas as pd, numpy as np, re, shutil

df = pd.read_csv("masoko_products_clean.csv")

# 1. Product type + brand fix for gaming items
def is_gift_card(name):
    n = str(name).strip()
    return n.startswith("$") or "gift card" in n.lower() or "wallet" in n.lower()

known_gaming_hw = {"Sony", "Playstation", "Nintendo", "Xbox", "Microsoft", "Logitech",
                   "Razer", "Steelseries", "Corsair", "Hyperx", "Havit", "Jbl", "JBL",
                   "Canyon", "Redragon", "Hama", "Thrustmaster", "Meta", "Oculus"}

def classify(row):
    if row["category_group"] != "gaming-items":
        return row["brand"], "Hardware/Accessory"
    if is_gift_card(row["name"]):
        return "Gift Card", "Gift Card"
    if row["brand"] in known_gaming_hw:
        return row["brand"], "Hardware/Accessory"
    return "Video Game", "Video Game"

res = df.apply(classify, axis=1, result_type="expand")
df["brand"], df["product_type"] = res[0], res[1]

# 2. Drop constant column
df = df.drop(columns=["in_stock"], errors="ignore")

# 3. Verify
print("Unique brands now:", df["brand"].nunique())
print("\nProduct types:\n", df["product_type"].value_counts())

print("\nGaming-items breakdown:")
g = df[df["category_group"] == "gaming-items"]
print(g["product_type"].value_counts())
print("Gaming 'Hardware' brands:", g[g["product_type"] == "Hardware/Accessory"]["brand"].value_counts().to_dict())

print("\nRemaining rare brands (count==1):")
vc = df["brand"].value_counts()
print(list(vc[vc == 1].index)[:30])

# 4. Duplicate name check
dups = df[df.duplicated("name", keep=False)].sort_values("name")
print("\nDuplicate-name rows:", len(dups), "| distinct SKUs among them:", dups["sku"].nunique())
print(dups[["name", "sku", "final_price"]].head(6).assign(name=lambda x: x["name"].str[:40]).to_string(index=False))

# 5. Stock cap check
print("\nstock_qty value counts (top):\n", df["stock_qty"].value_counts().head(6))

df.to_csv("masoko_products_clean.csv", index=False)
shutil.copy("masoko_products_clean.csv", "/content/drive/MyDrive/CodeAlpha/")
print("\nSaved.")

Unique brands now: 75

Product types:
 product_type
Hardware/Accessory    1207
Video Game              71
Gift Card               15
Name: count, dtype: int64

Gaming-items breakdown:
product_type
Video Game            71
Hardware/Accessory    46
Gift Card             15
Name: count, dtype: int64
Gaming 'Hardware' brands: {'Logitech': 24, 'Canyon': 10, 'Sony': 5, 'Playstation': 2, 'Nintendo': 2, 'Meta': 2, 'Hama': 1}

Remaining rare brands (count==1):
['10000Mah', 'Neon', 'Yeelight', 'Iwatch', 'Darth', 'Homepod', 'Swing', 'Glacier', 'Jellyfish', 'Ridge', 'Arch', 'Slicer', 'Honeycomb', 'Google', 'Dell', 'Lenovo', 'MAC', 'Amazon']

Duplicate-name rows: 74 | distinct SKUs among them: 74
                                    name        sku  final_price
AirPods Max 2 Wireless Over-Ear Headphon ACCPlug853      77836.0
AirPods Max 2 Wireless Over-Ear Headphon  ACCHot208      89097.0
          AirPods Pro 3 Wireless Earbuds ACCPlug561      33176.0
          AirPods Pro 3 Wireless Earbuds  ACCHo

In [ ]:

import pandas as pd, numpy as np, re, shutil

df = pd.read_csv("masoko_products_clean.csv")

# 1. Show the questionable rows BEFORE changing anything
check = ["Iwatch", "Homepod", "MAC", "10000Mah", "Darth", "Swing", "Glacier",
         "Jellyfish", "Ridge", "Arch", "Slicer", "Honeycomb"]
print(df[df["brand"].isin(check)][["brand", "name", "category_group"]]
        .assign(name=lambda x: x["name"].str[:50]).to_string(index=False))

# 2. Apply fixes
apple_fix = {"Iwatch", "Homepod", "MAC"}
legit_singletons = {"Google", "Dell", "Lenovo", "Amazon", "Neon", "Yeelight"}

df.loc[df["brand"].isin(apple_fix), "brand"] = "Apple"
vc = df["brand"].value_counts()
rare = set(vc[vc == 1].index) - legit_singletons
df.loc[df["brand"].isin(rare), "brand"] = "Other"

# 3. SKU prefix (listing channel hypothesis)
df["sku_prefix"] = df["sku"].astype(str).str.extract(r"^([A-Za-z]+)")[0]
print("\nSKU prefixes:\n", df["sku_prefix"].value_counts().head(10))

dups = df[df.duplicated("name", keep=False)]
g = dups.groupby("name")["final_price"].agg(["min", "max", "count"])
g["ratio"] = g["max"] / g["min"]
print("\nDuplicate groups:", len(g))
print("Median max/min price ratio:", g["ratio"].median().round(3))
print("Share of duplicate groups with different prices:", (g["ratio"] > 1.001).mean().round(2))
print("Prefixes among duplicates:\n", dups["sku_prefix"].value_counts().head(6))

# 4. Final verification
print("\nUnique brands:", df["brand"].nunique())
print(df["brand"].value_counts().head(15).to_string())

df.to_csv("masoko_products_clean.csv", index=False)
shutil.copy("masoko_products_clean.csv", "/content/drive/MyDrive/CodeAlpha/")
print("\nFinal shape:", df.shape, "| Saved.")

    brand                                               name category_group
 10000Mah 10000mAh UPS 9V 12V DC Uninterruptible Power Suppl    accessories
   Iwatch                              iWatch Charger Type C    accessories
     Arch                               Arch Headphone Stand    accessories
    Swing                              Swing Headphone Stand    accessories
  Glacier                    Glacier DualSense Display Stand    accessories
Jellyfish                          Jellyfish Headphone Stand    accessories
    Ridge                              Ridge Headphone Stand    accessories
    Darth                              Darth Vader PS5 Stand    accessories
   Slicer                       Slicer Xbox Controller Stand    accessories
Honeycomb                   Honeycomb Xbox Controller Holder    accessories
  Homepod                               HomePod mini - Black    accessories
      MAC      Mac Mini M4 Pro: 16GB RAM - 512GB SSD Storage  apple-devices

SKU prefixe

AttributeError: 'float' object has no attribute 'round'

In [ ]:

import pandas as pd, numpy as np, shutil

df = pd.read_csv("masoko_products_clean.csv")

# 1. Brand fixes
df.loc[df["brand"].isin({"Homepod", "MAC"}), "brand"] = "Apple"

legit_singletons = {"Google", "Dell", "Lenovo", "Amazon", "Neon", "Yeelight"}
vc = df["brand"].value_counts()
rare = set(vc[vc == 1].index) - legit_singletons
df.loc[df["brand"].isin(rare), "brand"] = "Other"

# 2. SKU split: product type code + source/channel code
sku_letters = df["sku"].astype(str).str.extract(r"^([A-Za-z]+)")[0]
df["sku_type"] = sku_letters.str[:3].str.upper()
df["sku_channel"] = sku_letters.str[3:]
print("SKU type codes:\n", df["sku_type"].value_counts().to_string())
print("\nSKU channels:\n", df["sku_channel"].value_counts().head(10).to_string())

# 3. Duplicate listings: do they differ by channel?
dups = df[df.duplicated("name", keep=False)]
g = dups.groupby("name")["final_price"].agg(["min", "max", "count"])
g["ratio"] = g["max"] / g["min"]
print("\nDuplicate groups:", len(g))
print("Median max/min price ratio:", round(float(g["ratio"].median()), 3))
print("Share with different prices:", round(float((g["ratio"] > 1.001).mean()), 2))

ch = dups.groupby("name")["sku_channel"].nunique()
print("Share of duplicate groups with different channels:", round(float((ch > 1).mean()), 2))
print("\nChannels among duplicates:\n", dups["sku_channel"].value_counts().head(6).to_string())

# 4. Final verification
print("\nUnique brands:", df["brand"].nunique())
print(df["brand"].value_counts().head(15).to_string())
print("Nulls in key columns:\n", df[["final_price", "brand", "product_type", "category_group"]].isna().sum().to_string())

df.to_csv("masoko_products_clean.csv", index=False)
shutil.copy("masoko_products_clean.csv", "/content/drive/MyDrive/CodeAlpha/")
print("\nFinal shape:", df.shape, "| Saved.")

SKU type codes:
 sku_type
ACC    712
DEV    342
GAM    134
LAP     74
SAP     16
SAT      8
ECO      5
HOM      2

SKU channels:
 sku_channel
Plug     494
Hot      262
ani      225
Sap       72
havit     60
Kyrpt     47
Lim       36
          31
Ama       23
Tru       14

Duplicate groups: 37
Median max/min price ratio: 1.031
Share with different prices: 1.0
Share of duplicate groups with different channels: 1.0

Channels among duplicates:
 sku_channel
Hot     36
Plug    34
ani      3
PHot     1

Unique brands: 64
brand
Apple         293
JBL           112
Video Game     71
Samsung        67
Hama           55
Havit          51
Xiaomi         45
Logitech       41
Canyon         40
Sony           40
Nothing        36
Oraimo         36
Marshall       30
Vision         28
Anker          27
Nulls in key columns:
 final_price       0
brand             0
product_type      0
category_group    0

Final shape: (1293, 27) | Saved.


In [ ]:

import pandas as pd, shutil

df = pd.read_csv("masoko_products_clean.csv")
df.loc[df["name"].str.startswith("Vision Plus"), "brand"] = "Vision Plus"

print("Rows:", len(df), "| Columns:", df.shape[1])
print("Brands:", df["brand"].nunique())
print("Discounted share:", round(df["has_discount"].mean() * 100, 1), "%")
print("Median final price KES:", df["final_price"].median())
print(df["category_group"].value_counts().to_string())

df.to_csv("masoko_products_clean.csv", index=False)
shutil.copy("masoko_products_clean.csv", "/content/drive/MyDrive/CodeAlpha/")
shutil.copy("masoko_products_raw.csv", "/content/drive/MyDrive/CodeAlpha/")
print("Saved both files to Drive")

Rows: 1293 | Columns: 27
Brands: 65
Discounted share: 24.7 %
Median final price KES: 17600.0
category_group
accessories      735
mobile-phones    299
gaming-items     132
laptops           74
apple-devices     53
Saved both files to Drive
